# Combined Autoencoder Anomaly Detection for TLS Profiling

This notebook evaluates the `build_combined_autoencoder` architecture from `tls_profiling.autoencoder.models`. The dataset paths are parameterized so the same experiment can be run on one or more parquet partitions, and the target categories are controlled through `category_labels`. For each target label, the model is trained only on flows from that label and then evaluated on its ability to separate that label from the rest of the traffic using reconstruction error as the anomaly score.


### PARAMETERS:


In [12]:
train_features_path = ["../data-ext/malware_train.parquet", "../data-ext/apps_train.parquet"]
val_features_path = ["../data-ext/malware_val.parquet", "../data-ext/apps_val.parquet"]
test_features_path = ["../data-ext/malware_test.parquet", "../data-ext/apps_test.parquet"]
train_labels_path = ["../data-ext/malware_train_labels.parquet", "../data-ext/apps_train_labels.parquet"]
val_labels_path = ["../data-ext/malware_val_labels.parquet", "../data-ext/apps_val_labels.parquet"]
test_labels_path = ["../data-ext/malware_test_labels.parquet", "../data-ext/apps_test_labels.parquet"]
category_labels = ["system", "malware", "application"]
threshold_quantile = 0.999


## Full-Feature Evaluation

Unlike `ae_malware_base.ipynb`, this notebook does not run feature ablations. It always uses the full preprocessed feature vector.

To match the contract of `build_combined_autoencoder`, the transformed dataframe is explicitly reordered before conversion to NumPy:

- all `tls.rec.*` columns are placed first and become the sequence/convolution branch
- all remaining columns are placed after them and become the tabular branch

This guarantees that the internal split performed by the model builder matches the intended semantics of the inputs.


## Evaluation

For each label listed in `category_labels`, that label is treated as the in-profile or normal class. All remaining labels are treated as anomalies.

The evaluation uses the same three disjoint splits as the baseline notebooks:

- `train`: only samples from the selected normal class, used to fit the autoencoder
- `validation`: clean in-profile traffic, used to set the anomaly-score threshold as a high quantile of reconstruction error
- `test`: held-out mixed traffic, used only for final reporting

Because the neural model still needs a monitoring split during fitting, the notebook carves a small internal holdout only from the selected normal training traffic and uses the public validation split only on clean in-profile samples for threshold calibration.


In [13]:
import sys
sys.path.append('../../src')


In [14]:
FLOW = ['bs', 'ps', 'br', 'pr', 'td']
CTLS = ['tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv']
STLS = ['tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv']
REC = ['tls.rec']
FULL_FEATURE_SET = FLOW + CTLS + STLS + REC


In [15]:
from pathlib import Path

import pandas as pd
from tls_profiling.preprocessing import build_and_fit_preprocessor

def read_parquet_files(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    return pd.concat([pd.read_parquet(Path(path)) for path in paths], ignore_index=True)

print('Loading extracted features from parameterized parquet paths.')
df_train = read_parquet_files(train_features_path)
df_val = read_parquet_files(val_features_path)
df_test = read_parquet_files(test_features_path)
df_train_label = read_parquet_files(train_labels_path)
df_val_label = read_parquet_files(val_labels_path)
df_test_label = read_parquet_files(test_labels_path)

preprocessor = build_and_fit_preprocessor(df_train)
print('Built preprocessor from df_train.')


Loading extracted features from parameterized parquet paths.
Built preprocessor from df_train.


In [16]:
import random

import numpy as np
import tensorflow as tf
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split

from tls_profiling.autoencoder import train_autoencoder_model, compute_reconstruction_error
from tls_profiling.autoencoder.models import build_combined_autoencoder

SEED = 1313
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AE_CONFIG = {
    'encoding_dim': 16,
    'conv_filters': (32, 64),
    'conv_kernel_sizes': (3, 3),
    'conv_pool_sizes': (2, None),
    'dense_branch_units': (64, 64),
    'shared_encoder_units': (64,),
    'seq_decoder_units': (64,),
    'meta_decoder_units': (64,),
    'activation': 'relu',
    'pooling': 'global_max',
    'norm_type': None,
    'dropout_rate': 0.0,
    'l2_reg': 0.0,
    'max_epochs': 50,
    'batch_size': 128,
    'lr': 1e-3,
    'loss': 'mse',
    'early_stopping_patience': 8,
    'train_holdout_ratio': 0.2,
}

def choose_quantile_threshold(anomaly_score, quantile):
    if len(anomaly_score) == 0:
        raise ValueError('Cannot choose a threshold from an empty score array.')
    return float(np.quantile(anomaly_score, quantile))

def to_numpy_float32(x):
    return x.to_numpy(dtype=np.float32, copy=False)

def split_train_for_autoencoder(x_train):
    if len(x_train) < 3:
        return x_train, x_train

    x_fit, x_monitor = train_test_split(
        x_train,
        test_size=AE_CONFIG['train_holdout_ratio'],
        random_state=SEED,
        shuffle=True,
    )
    return x_fit, x_monitor

def choose_encoding_dim(input_dim):
    return max(2, min(AE_CONFIG['encoding_dim'], max(2, input_dim // 2)))

def reorder_for_combined_autoencoder(x):
    seq_columns = sorted(column for column in x.columns if column.startswith('tls.rec.'))
    tabular_columns = sorted(column for column in x.columns if not column.startswith('tls.rec.'))

    if not seq_columns:
        raise ValueError('No tls.rec.* columns found after preprocessing.')
    if not tabular_columns:
        raise ValueError('No non-sequence columns found after preprocessing.')

    ordered_columns = seq_columns + tabular_columns
    return x.loc[:, ordered_columns], seq_columns, tabular_columns

def prepare_full_feature_matrix(x):
    transformed = preprocessor.transform(x)
    reordered, seq_columns, tabular_columns = reorder_for_combined_autoencoder(transformed)
    return reordered, seq_columns, tabular_columns

evaluation_cache = {}

def evaluate(normal_label):
    if normal_label in evaluation_cache:
        return evaluation_cache[normal_label]

    x_train = df_train.loc[df_train_label['category'] == normal_label].reset_index(drop=True)
    if len(x_train) == 0:
        raise ValueError(
            f"No training rows found for normal_label={normal_label!r}. "
            "Update category_labels or dataset paths."
        )
    x_val_normal = df_val.loc[df_val_label['category'] == normal_label].reset_index(drop=True)
    if len(x_val_normal) == 0:
        raise ValueError(
            f"No validation rows found for normal_label={normal_label!r}. "
            "Update category_labels or dataset paths."
        )
    x_test = df_test
    y_test = (df_test_label['category'] != normal_label).astype(int).to_numpy()
    test_metadata = df_test_label[['category', 'label']].reset_index(drop=True).copy()

    x_train_transformed, seq_columns, tabular_columns = prepare_full_feature_matrix(x_train)
    x_val_normal_transformed, _, _ = prepare_full_feature_matrix(x_val_normal)
    x_test_transformed, _, _ = prepare_full_feature_matrix(x_test)

    x_train_np = to_numpy_float32(x_train_transformed)
    x_val_normal_np = to_numpy_float32(x_val_normal_transformed)
    x_test_np = to_numpy_float32(x_test_transformed)

    x_fit, x_monitor = split_train_for_autoencoder(x_train_np)

    input_dim = x_fit.shape[1]
    encoding_dim = choose_encoding_dim(input_dim)
    conv_input_size = len(seq_columns)

    tf.keras.backend.clear_session()

    models = build_combined_autoencoder(
        input_dim=input_dim,
        encoding_dim=encoding_dim,
        conv_input_size=conv_input_size,
        conv_filters=AE_CONFIG['conv_filters'],
        conv_kernel_sizes=AE_CONFIG['conv_kernel_sizes'],
        conv_pool_sizes=AE_CONFIG['conv_pool_sizes'],
        dense_branch_units=AE_CONFIG['dense_branch_units'],
        shared_encoder_units=AE_CONFIG['shared_encoder_units'],
        seq_decoder_units=AE_CONFIG['seq_decoder_units'],
        meta_decoder_units=AE_CONFIG['meta_decoder_units'],
        activation=AE_CONFIG['activation'],
        pooling=AE_CONFIG['pooling'],
        norm_type=AE_CONFIG['norm_type'],
        dropout_rate=AE_CONFIG['dropout_rate'],
        l2_reg=AE_CONFIG['l2_reg'],
        seq_output_activation='sigmoid',
        meta_output_activation='sigmoid',
    )

    history = train_autoencoder_model(
        models=models,
        x_train=x_fit,
        x_val=x_monitor,
        max_epochs=AE_CONFIG['max_epochs'],
        batch_size=AE_CONFIG['batch_size'],
        lr=AE_CONFIG['lr'],
        loss=AE_CONFIG['loss'],
        early_stopping_patience=AE_CONFIG['early_stopping_patience'],
        verbose=0,
    )

    val_normal_scores = compute_reconstruction_error(models.autoencoder, x_val_normal_np, metric='mse')
    threshold = choose_quantile_threshold(val_normal_scores, threshold_quantile)
    anomaly_score = compute_reconstruction_error(models.autoencoder, x_test_np, metric='mse')

    result = {
        'y_test': y_test,
        'anomaly_score': anomaly_score,
        'threshold': threshold,
        'test_metadata': test_metadata,
        'history': history,
        'input_dim': input_dim,
        'encoding_dim': encoding_dim,
        'conv_input_size': conv_input_size,
        'seq_feature_count': len(seq_columns),
        'tabular_feature_count': len(tabular_columns),
        'ordered_columns': list(x_train_transformed.columns),
    }
    evaluation_cache[normal_label] = result
    return result

def debug_csv(normal_label, y_test, y_pred, anomaly_score, test_metadata):
    output_dir = Path('tmp')
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / f'ae_combined_{normal_label}_full.csv'
    debug_df = test_metadata.rename(columns={'category': 'original_category', 'label': 'original_label'}).copy()
    debug_df['y_test'] = y_test
    debug_df['y_pred'] = y_pred
    debug_df['anomaly_score'] = anomaly_score
    debug_df.to_csv(output_path, index=False)

def compute_scores(normal_label):
    result = evaluate(normal_label)
    y_test = result['y_test']
    anomaly_score = result['anomaly_score']
    threshold = result['threshold']
    test_metadata = result['test_metadata']

    auc = roc_auc_score(y_test, anomaly_score)
    ap = average_precision_score(y_test, anomaly_score)
    y_pred = (anomaly_score >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)

    debug_csv(normal_label, y_test, y_pred, anomaly_score, test_metadata)
    return {
        'auc_score': auc,
        'ap_score': ap,
        'f1_score': f1,
        'threshold': threshold,
        'input_dim': result['input_dim'],
        'encoding_dim': result['encoding_dim'],
        'conv_input_size': result['conv_input_size'],
        'seq_feature_count': result['seq_feature_count'],
        'tabular_feature_count': result['tabular_feature_count'],
    }

def plot_curves(normal_label):
    result = evaluate(normal_label)
    y_test = result['y_test']
    anomaly_score = result['anomaly_score']

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        anomaly_score,
        name='Combined AE PR-AUC',
        ax=axes[0],
    )
    axes[0].set_title(f'{normal_label} Precision-Recall')

    RocCurveDisplay.from_predictions(
        y_test,
        anomaly_score,
        name='Combined AE AUC-ROC',
        ax=axes[1],
    )
    axes[1].set_title(f'{normal_label} ROC Curve')

    plt.tight_layout()
    plt.show()


In [17]:
sample_features, sample_seq_columns, sample_tabular_columns = prepare_full_feature_matrix(df_train.head(5))
print(f'Full feature count: {sample_features.shape[1]}')
print(f'Sequence feature count: {len(sample_seq_columns)}')
print(f'Tabular feature count: {len(sample_tabular_columns)}')
print('First sequence columns:', sample_seq_columns[:5])
print('First tabular columns:', sample_tabular_columns[:5])
display(sample_features.head())


Full feature count: 178
Sequence feature count: 20
Tabular feature count: 158
First sequence columns: ['tls.rec.0', 'tls.rec.1', 'tls.rec.10', 'tls.rec.11', 'tls.rec.12']
First tabular columns: ['br', 'bs', 'pr', 'ps', 'td']


,tls.rec.0,tls.rec.1,tls.rec.10,tls.rec.11,tls.rec.12,tls.rec.13,tls.rec.14,tls.rec.15,tls.rec.16,tls.rec.17,...,tls.ssv.,tls.ssv.0304,tls.ssv.grease,tls.ssv.other,tls.ssv.unusual,tls.sver_,tls.sver_0300,tls.sver_0301,tls.sver_0302,tls.sver_0303
0,0.782948,0.372027,0.233757,0.201641,0.463245,0.218991,0.206484,0.202031,0.201296,0.203379,...,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0
1,0.778445,0.382007,0.227642,0.086106,0.439459,0.212741,0.110518,0.211634,0.195551,0.108856,...,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0
2,0.778445,0.382007,0.227542,0.107709,0.435041,0.212741,0.111737,0.211634,0.195845,0.158624,...,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0
3,0.782948,0.372027,0.234698,0.247174,0.427986,0.218991,0.206484,0.202031,0.201296,0.203379,...,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0
4,0.782948,0.372027,0.234698,0.205878,0.428320,0.218991,0.206484,0.202031,0.201296,0.203379,...,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0


In [18]:
rows = []

for label in category_labels:
    display(f"Scoring {label}...")
    scores = compute_scores(label)
    rows.append({
        'FeatureSet': 'Full',
        'ClassLabel': label,
        'AucScore': scores['auc_score'],
        'ApScore': scores['ap_score'],
        'F1Score': scores['f1_score'],
        'Threshold': scores['threshold'],
        'InputDim': scores['input_dim'],
        'EncodingDim': scores['encoding_dim'],
        'SequenceFeatureCount': scores['seq_feature_count'],
        'TabularFeatureCount': scores['tabular_feature_count'],
        'ConvInputSize': scores['conv_input_size'],
    })

results_df = pd.DataFrame(
    rows,
    columns=[
        'FeatureSet',
        'ClassLabel',
        'AucScore',
        'ApScore',
        'F1Score',
        'Threshold',
        'InputDim',
        'EncodingDim',
        'SequenceFeatureCount',
        'TabularFeatureCount',
        'ConvInputSize',
    ],
)
display(results_df)


'Scoring system...'

'Scoring malware...'

'Scoring application...'

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score,Threshold,InputDim,EncodingDim,SequenceFeatureCount,TabularFeatureCount,ConvInputSize
0,Full,system,0.955084,0.964464,0.002103,4.679806,178,16,20,158,20
1,Full,malware,0.698074,0.657638,0.004348,1.792144,178,16,20,158,20
2,Full,application,0.521434,0.985576,0.000028,17.074970,178,16,20,158,20


## System Profile

The table below isolates the combined autoencoder result for the `system` profile using the full feature set.


In [8]:
system_df = results_df[results_df['ClassLabel'] == 'system'].sort_values('F1Score', ascending=False)
display(system_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score,Threshold,InputDim,EncodingDim,SequenceFeatureCount,TabularFeatureCount,ConvInputSize
0,Full,system,0.955084,0.964464,0.94091,0.000023,178,16,20,158,20


## Malware Profile

This section shows the full-feature combined autoencoder evaluation when `malware` is treated as the in-profile class.


In [9]:
malware_df = results_df[results_df['ClassLabel'] == 'malware'].sort_values('F1Score', ascending=False)
display(malware_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score,Threshold,InputDim,EncodingDim,SequenceFeatureCount,TabularFeatureCount,ConvInputSize
1,Full,malware,0.698074,0.657638,0.779363,0.000025,178,16,20,158,20


## Application Profile

This section shows the full-feature combined autoencoder evaluation when `application` is treated as the in-profile class.


In [10]:
application_df = results_df[results_df['ClassLabel'] == 'application'].sort_values('F1Score', ascending=False)
display(application_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score,Threshold,InputDim,EncodingDim,SequenceFeatureCount,TabularFeatureCount,ConvInputSize
2,Full,application,0.521434,0.985576,0.994414,0.000005,178,16,20,158,20
